In [43]:
import numpy as np
import pandas as pd
import cvxpy as cp
from dataclasses import dataclass

In [44]:
@dataclass
class Network:
    zones: list           # zone names, length Z
    zone_of: np.ndarray   # (N,) zone index of each node
    lines: list           # [(from_node, to_node)], length L, 0-based node indices
    x: np.ndarray         # (L,) reactance [p.u.]
    f_max: np.ndarray     # (L,) thermal limit [MW]
    slack: int            # slack node index
    g_max: np.ndarray     # (N,) generator capacity [MW]
    c_gen: np.ndarray     # (N,) generation cost [€/MWh]
    c_curt: np.ndarray    # (N,) RES curtailment penalty [€/MWh]
   # c_adj: np.ndarray     # (N,) intraday adjustment cost [€/MWh]
    c_red: np.ndarray     # (N,) redispatch cost [€/MWh]

    @property
    def N(self): return len(self.zone_of)
    @property
    def L(self): return len(self.lines)
    @property
    def Z(self): return len(self.zones)

In [45]:
def ex_network(f_max=(100.0, 100.0, 100.0, 100.0)):
    lines = [(0, 1), (0, 2), (0, 3), (2, 3)]
    g_max = np.array([20.0, 20.0, 10.0, 25.0])
    N = len(g_max)   # one generator per node

    return Network(
        zones=["1", "2", "3"],
        zone_of=np.array([0, 0, 1, 2]),
        lines=lines, x=np.ones(len(lines)), f_max=np.array(f_max), slack=1,
        g_max=g_max,
        c_gen=np.array([9.0, 20.0, 10.0, 11.0]),
        c_curt=np.full(N, 10.0),
        c_red=np.full(N, 5.0),
    )

In [46]:
def incidence_matrix(net):
    A = np.zeros((net.L, net.N))
    for l, (i, j) in enumerate(net.lines):
        A[l, i], A[l, j] = 1.0, -1.0
    return A


def ptdf_matrix(net):
    A = incidence_matrix(net)
    b = 1.0 / net.x
    B_line = np.diag(b) @ A
    B_bus = A.T @ np.diag(b) @ A
    keep = [n for n in range(net.N) if n != net.slack]
    X_bus = np.zeros((net.N, net.N))
    X_bus[np.ix_(keep, keep)] = np.linalg.inv(B_bus[np.ix_(keep, keep)])
    return B_line @ X_bus


def zone_mapping(net):
    m = np.zeros((net.Z, net.N))
    m[net.zone_of, np.arange(net.N)] = 1.0
    return m

In [55]:
net_3 = ex_network()
net_3.lines
np.diag(1/net_3.x) @A

array([[ 1., -1.,  0.,  0.],
       [ 1.,  0., -1.,  0.],
       [ 1.,  0.,  0., -1.],
       [ 0.,  0.,  1., -1.]])

In [50]:
net_3 = ex_network()
A = incidence_matrix(net_3)
ptdf = ptdf_matrix(net_3)

In [56]:
A

array([[ 1., -1.,  0.,  0.],
       [ 1.,  0., -1.,  0.],
       [ 1.,  0.,  0., -1.],
       [ 0.,  0.,  1., -1.]])

In [51]:
ptdf

array([[ 1.00000000e+00,  0.00000000e+00,  1.00000000e+00,
         1.00000000e+00],
       [ 2.22044605e-16,  0.00000000e+00, -6.66666667e-01,
        -3.33333333e-01],
       [ 1.11022302e-16,  0.00000000e+00, -3.33333333e-01,
        -6.66666667e-01],
       [-1.11022302e-16,  0.00000000e+00,  3.33333333e-01,
        -3.33333333e-01]])